<a href="https://colab.research.google.com/github/hasana1atwit/Movie-Analysis/blob/main/Movie_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSV Import
To access csv data file, first upload to google drive, then run the following code:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


After granting colab access to your files, copy the file path and run the next cell to read the data csv.

In [ ]:
import pandas as pd
file_path = '/content/drive/MyDrive/movies_dataset.csv' # Paste your path here using the Files tab on the left hand side
movies = pd.read_csv(file_path)

Characteristics of the data:

In [ ]:
movie_df = pd.DataFrame(movies)
movie_df.head()

,MovieID,Title,Genre,ReleaseYear,ReleaseDate,Country,BudgetUSD,US_BoxOfficeUSD,Global_BoxOfficeUSD,Opening_Day_SalesUSD,One_Week_SalesUSD,IMDbRating,RottenTomatoesScore,NumVotesIMDb,NumVotesRT,Director,LeadActor
0,1,Might toward capital,Comedy,2003,28-09-2003,China,6577427.79,6613685.82,15472035.66,1778530.85,3034053.32,6.2,58,7865,10596,Kristina Moore,Brian Mccormick
1,2,He however experience,Comedy,1988,14-02-1988,USA,1883810.10,1930949.15,3637731.12,247115.74,831828.84,5.2,44,1708,220,Benjamin Hudson,Ashley Pena
2,3,Star responsibility politics,Comedy,1971,02-11-1971,USA,2468079.29,4186694.69,7165111.24,878453.95,2171405.93,5.5,55,4678,7805,Kayla Young,Alexander Haley
3,4,Exactly live,Comedy,1998,06-08-1998,USA,1447311.46,2023683.92,4373820.26,570657.72,898886.01,7.3,87,2467,1751,Michael Ross,Patrick Barnett
4,5,Focus improve especially,Documentary,2021,17-12-2021,India,900915.86,2129629.10,3113017.38,361189.37,861775.91,6.1,67,5555,697,Faith Franklin,Duane Fletcher DDS


In [ ]:
movie_df.shape

(999999, 17)

# 1. Does a movie's budget affect its success?

In [ ]:
import matplotlib.pyplot as plt

#scatter plot budget
x = movies['BudgetUSD']
y1 = movies['US_BoxOfficeUSD']
y2 = movies['Global_BoxOfficeUSD']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.plot(x,y1)
ax1.set_xlabel('Budget (USD)')
ax1.set_ylabel('US Box Office (USD)')
ax1.set_title('Budget vs. US Box Office')

ax2.plot(x,y2)
ax2.set_xlabel('Budget (USD)')
ax2.set_ylabel('Global Box Office (USD)')
ax2.set_title('Budget vs. Global Box Office')

plt.show()

Statistical Approach

In [ ]:
# US Box Office p_value
import statsmodels.api as sm

x = sm.add_constant(x)
statModel1 = sm.OLS(y1, x).fit()
#print(model.summary())
statModel1.pvalues

In [ ]:
# Global Box Office_pvalue
import statsmodels.api as sm

x = sm.add_constant(x)
statModel2 = sm.OLS(y2, x).fit()
#print(model.summary())
statModel2.pvalues

In [ ]:
# US Box Office R^2
r1 = statModel1.rsquared
r1

In [ ]:
# Global Box Office R^2
r2 = statModel2.rsquared
r2

In [ ]:
# US Box Office Parameters
statModel1.params

In [ ]:
# Global Box office Parameters
statModel2.params

Linear Regression (ML) Technique

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import copy

df_movieBudget = pd.DataFrame({
    'BudgetUSD': movie_df['BudgetUSD'],
    'US_BoxOfficeUSD': movie_df['US_BoxOfficeUSD'],
    'Global_BoxOfficeUSD': movie_df['Global_BoxOfficeUSD']
})

df_movieBudget = df_movieBudget.sample(frac=1, random_state=42)

In [ ]:
# from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


X = df_movieBudget[['BudgetUSD']]
y = df_movieBudget['US_BoxOfficeUSD']

# Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_test_pred = model.predict(X_test)

# Evaluate
print("R²:", model.score(X_test, y_test))
print("MSE:", mean_squared_error(y_test, y_test_pred))


R²: 0.7844421447896011
MSE: 324920596078786.9


In [ ]:
# from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


X = df_movieBudget[['BudgetUSD']]
y = df_movieBudget['Global_BoxOfficeUSD']

# Split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Predict
y_test_pred = model.predict(X_test)

# Evaluate
print("R²:", model.score(X_test, y_test))
print("MSE:", mean_squared_error(y_test, y_test_pred))


R²: 0.8060951939367172
MSE: 945048749645301.9


# 2. Does the majority of profit come from the country it was made in or internationally?


In [ ]:
import seaborn as sns

cols_needed = ['Country', 'US_BoxOfficeUSD', 'Global_BoxOfficeUSD', 'BudgetUSD', 'ReleaseYear']
df = pd.read_csv('movies_dataset.csv', usecols=cols_needed)

#filter USA movies
df = df[df['Country'] == 'USA']

#optomize oversized data
for col in ['US_BoxOfficeUSD', 'Global_BoxOfficeUSD', 'BudgetUSD']:
    df[col] = pd.to_numeric(df[col], errors='coerce', downcast='float')

df['ReleaseYear'] = pd.to_numeric(df['ReleaseYear'], errors='coerce', downcast='integer')

df = df.dropna()

In [ ]:
#Create international Revenue value
df['International_BoxOfficeUSD'] = df['Global_BoxOfficeUSD'] - df['US_BoxOfficeUSD']

# Remove invalid data
df = df[
    (df['BudgetUSD'] > 0) &
    (df['US_BoxOfficeUSD'] > 0) &
    (df['International_BoxOfficeUSD'] > 0)
]

In [ ]:
total_domestic = df['US_BoxOfficeUSD'].sum()
total_international = df['International_BoxOfficeUSD'].sum()

print(f"\nAnalysis of {len(df)} USA-made movies\n")
print(f"Total Domestic:      ${total_domestic:,.2f}")
print(f"Total International: ${total_international:,.2f}")

In [ ]:
df['Total_Revenue'] = df['US_BoxOfficeUSD'] + df['International_BoxOfficeUSD']

df = df[df['Total_Revenue'] > 0]

df['Domestic_Share'] = df['US_BoxOfficeUSD'] / df['Total_Revenue']
df['International_Share'] = df['International_BoxOfficeUSD'] / df['Total_Revenue']

yearly_share = df.groupby('ReleaseYear', as_index=False)[
    ['Domestic_Share', 'International_Share']
].mean()

plt.figure(figsize=(10, 6))
plt.bar(yearly_share['ReleaseYear'], yearly_share['Domestic_Share'], label='Domestic')
plt.bar(
    yearly_share['ReleaseYear'],
    yearly_share['International_Share'],
    bottom=yearly_share['Domestic_Share'],
    label='International'
)
plt.legend()
plt.title('Revenue Share Over Time')
plt.show()

In [ ]:
wins = (df['International_BoxOfficeUSD'] > df['US_BoxOfficeUSD']).value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    wins,
    labels=['International > Domestic', 'Domestic > International'],
    autopct='%1.1f%%'
)
plt.title('Which Market Wins Per Movie')
plt.show()

# 3. Do shorter movie titles make more money?

In [ ]:
import matplotlib.pyplot as plt

#create a new column for the lengths of the movie titles
movies['Title_len'] = movies['Title'].str.len()

In [ ]:
#scatter plot title length vs. total gross
plt.scatter(movies['Title_len'], movies['Global_BoxOfficeUSD'] / 10000000)
plt.title('Length of Movie Title vs. Total Gross Globally')
plt.xlabel('Title Length')
plt.ylabel('Total Gross Globally (in Billions)')
plt.show()

In [ ]:
# the scatter plot was too difficult to assess at a glance
# create groups and plot group means instead

# add new column for mean total gross by title length
movies['Gross_by_title_len'] = movies.groupby('Title_len')['Global_BoxOfficeUSD'].mean()

# plot grouped title lengths and total gross
plt.plot(movies.groupby('Title_len')['Global_BoxOfficeUSD'].mean(), marker='o')
plt.title('Length of Movie Title (grouped) vs. Total Gross Globally')
plt.xlabel('Title Length')
plt.ylabel('Total Gross Globally (in Billions)')
plt.show()

In [ ]:
# even further, create categories of lengths to compare averages
# categorical data can be analyzed using one-way ANOVA

import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

# create categories
# max(movies['Title_len']) = 41, so categories should stop around then
bins = [1, 6, 11, 16, 21, 26, 31, 36, 41, 45] # values to portion out categories
labels = ['1-5', '6-10', '11-15', '16-20', '21-25', '26-30', '31-35', '36-40', '41-45']

movies['Title_len_5s'] = pd.cut(movies['Title_len'], bins=bins, labels=labels, right=False) # new column of titles grouped by fives

# show distribution
plt.figure(figsize=(10,6))
plt.bar(movies['Title_len_5s'].value_counts().sort_index().index, movies['Title_len_5s'].value_counts().sort_index().values)
plt.xlabel('Title Length Group')
plt.ylabel('Frequency')
plt.title('Distribution of Movie Title Lengths')
plt.show()

In [ ]:
# define the OLS model for ANOVA use
model = ols('Global_BoxOfficeUSD ~ C(Title_len_5s)', data=movies).fit()

# perform the ANOVA and print the table
anova_table = sm.stats.anova_lm(model, typ=1)
print(anova_table)

The p-value of the one-way ANOVA is 0.258998. The null hypothesis is that the length of movie titles does not affect the total box office revenue. Since the resulting p-value is greater than 0.05, we fail to reject the null hypothesis. This means that there is no correlation between the lengths of movie titles and their total gross.

To ensure the accuracy of the ANOVA, we must test the following assumptions: independence, normality, homoscedasticity.

1. Indepedence: by design of the dataset, we can confirm that the samples are independent.
2. Normality of Residuals: the QQ plot below shows that the residuals may not normally distributed because the values do not follow a straight line.
3. Homoscedasticity: the variances between groups are equal, as shown by the horizontal line in the residuals vs. fitted values plot below. If the normality assumption hadn't been violated, then this plot would have been centered around y=0.

In [ ]:
# QQ plot to test normality of residuals
resid = model.resid
sm.qqplot(resid, line='s')
plt.title("Q-Q Plot of Residuals")
plt.show()

# residuals vs. fitted values plot to test equal variance
fitted = model.fittedvalues
plt.scatter(resid, fitted, marker='o')
plt.title('Residual vs. Fitted Values')
plt.xlabel('Residual Values')
plt.ylabel('Fitted Values')
plt.show()

#4. Does release season affect box office performance?

In [ ]:
# categorize release dates by season (winter: Dec-Feb, spring: Mar-May, summer: Jun-Aug, fall: Sep-Nov)
month_map = {
    1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring',
    5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn', 12: 'Winter'
}
movies['Date'] = pd.to_datetime(movies['ReleaseDate'], dayfirst=True) # convert String values to DateTimes
movies['Season'] = movies['Date'].dt.month.map(month_map) # change DateTimes to seasons

In [ ]:
# define the OLS model for ANOVA use
model = ols('Global_BoxOfficeUSD ~ C(Season)', data=movies).fit()

# perform the ANOVA and print the table
anova_table = sm.stats.anova_lm(model, typ=1)
print(anova_table)

The p-value in this ANOVA test is 0.968961, which is greater than 0.05. Fail to reject the null hypothesis. There is no effect between release season and box office success.

In [ ]:
# ANOVA assumptions
# 1. Independence: each movie can only be released in one season, so there is no overlap between categories.

# 2. Normality of residuals:
# QQ plot to test normality of residuals
resid = model.resid
sm.qqplot(resid, line='s')
plt.title("Q-Q Plot of Residuals")
plt.show()

# 3. Homoscedasticity:
# residuals vs. fitted values plot to test equal variance
fitted = model.fittedvalues
plt.scatter(resid, fitted, marker='o')
plt.title('Residual vs. Fitted Values')
plt.xlabel('Residual Values')
plt.ylabel('Fitted Values')
plt.show()

#5 Which Genre has the most influence on success?

In [ ]:
# Data Cleaning
# Convert revenue to numeric and drop rows with missing values
df['Global_BoxOfficeUSD'] = pd.to_numeric(df['Global_BoxOfficeUSD'], errors='coerce')
df = df.dropna(subset=['Global_BoxOfficeUSD', 'Genre'])

# Calculate Statistics
genre_stats = df.groupby('Genre')['Global_BoxOfficeUSD'].agg(['sum', 'mean', 'median', 'count']).reset_index()

In [ ]:
# GRAPH 1: Total Global Revenue (The "Market Share")
plt.figure(figsize=(12, 8))
total_sorted = genre_stats.sort_values(by='sum', ascending=False)

ax1 = sns.barplot(data=total_sorted, x='sum', y='Genre', palette='viridis')

# Add labels for total revenue
for i, row in total_sorted.iterrows():
    val = row['sum']
    label = f'${val/1e9:.2f}B' if val >= 1e9 else f'${val/1e6:.1f}M'
    # .get_loc helps find the correct vertical position after sorting
    ax1.text(val, total_sorted.index.get_loc(i), f' {label}', va='center', fontweight='bold')

plt.title('Total Global Revenue by Genre (All Countries)', fontsize=15)
plt.xlabel('Total Revenue (USD)')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# GRAPH 2: Mean vs. Median
plt.figure(figsize=(12, 8))
avg_sorted = genre_stats.sort_values(by='mean', ascending=False)

# Prepare data for grouped bar chart
melted_stats = avg_sorted.melt(id_vars='Genre', value_vars=['mean', 'median'],
                               var_name='Metric', value_name='Revenue')

ax2 = sns.barplot(data=melted_stats, x='Revenue', y='Genre', hue='Metric', palette='coolwarm')

# Add labels
for p in ax2.patches:
    width = p.get_width()
    if width > 0:
        label = f'${width/1e9:.1f}B' if width >= 1e9 else f'${width/1e6:.1f}M'
        ax2.text(width, p.get_y() + p.get_height()/2, f' {label}', va='center', fontsize=9)

plt.title('Average (Mean) vs. Typical (Median) Revenue per Genre', fontsize=15)
plt.xlabel('Revenue (USD)')
plt.tight_layout()

plt.show()